# Lab 09-2 - Boosting Ensembles and `xgboost`

This lab introduces gradient boosting methods with a focus on the `xgboost` library.  

Currently, `xgboost` is viewed as a state-of-the-art machine learning algorithm for flat/tabular data. There exist other more advanced gradient boosting libraries, but we will not cover these models in this lab because of their heavy overlap with `xgboost`.

Unlike the libraries we've been using so far, `xgboost` is not a default inclusion in Anaconda, so you'll need to install it yourself. To do so, first run the following code cell:

Then restart the kernel (click "Restart" in the notebook toolbar). If you have `xgboost` installed, you should be able to load the library below without an error. Once you've successfully installed it, delete the code block above that installs it. 

In [ ]:
import warnings   # We will turn off the future warnings that xgboost gives us
warnings.simplefilter(action = 'ignore', category = FutureWarning)
import xgboost as xgb

We'll also need to load our standard libraries:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

The examples in this lab will use data from a public health study conducted in Araihazar upazila, Bangladesh. In this study, researchers tested water from community wells for arsenic and encouraged households that were using unsafe wells to switch to a nearby well that was tested to be safe. Several years after their initial testing, the researchers revisited each household they had encouraged to switch to a safe well and recorded which of these households had done so, a $y$ binary outcome of `switch = 'yes'`.

In [ ]:
# Read data
wells = pd.read_csv(filepath_or_buffer = "../data/Wells.csv")
wells.head()


In [ ]:
# EDA: histogram of the outcome variable switch. Around 57.5% yes
plt.hist(x = wells['switch'])
plt.xlabel(xlabel = 'switch')
plt.ylabel(ylabel = 'Number of households')
plt.show()

In [ ]:
# Train-test split
from sklearn.model_selection import train_test_split
train, test = train_test_split(wells, test_size = 0.1, random_state = 76)

# Separate the outcome, converting 'yes'/'no' to 1/0 integer
train_y = (train['switch'] == 'yes').astype(int)
test_y = (test['switch'] == 'yes').astype(int)

# Separate the features, dropping "association"
train_X = train.drop(labels = ['switch', 'association'], axis = "columns")
test_X = test.drop(labels = ['switch', 'association'], axis = 'columns')
train_X.head()

## A) Boosting in `sklearn`

The *AdaBoost algorithm* is regarded as the first successful boosting method. Modern gradient boosting approaches are generalizations of AdaBoost, making it worthwhile to have some familiarity with AdaBoost for its historical importance and easy to understand framework.

AdaBoost sequentially builds an ensemble classifier using decision trees. The first tree in the ensemble gives each data-point a weight of $\tfrac{1}{n}$, as is normal. However, the next tree assigns each data-point a weight that is determined by its prediction error, thereby allowing data-points that are misclassified to be given more attention. 


An essential parameter in the AdaBoost algorithm is the *learning rate*, which controls how much weight is given to a base model at each boosting iteration. The default value is 1, which allows each base model to contribute proportional to its error:

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

# Setup two different AdaBoost models with two different learning rates
ada_model1 = AdaBoostClassifier(n_estimators = 100, learning_rate = 1, random_state = 76)
ada_model2 = AdaBoostClassifier(n_estimators = 100, learning_rate = 0.2, random_state = 76)

# Plot weights of each tree in the ensemble:
x = np.linspace(start = 1, stop = 101, num = 100)
y1 = ada_model1.fit(X = train_X, y = train_y).estimator_weights_
y2 = ada_model2.fit(X = train_X, y = train_y).estimator_weights_

plt.plot(x, y1, label = "learning_rate = 1")
plt.plot(x, y2, label = "learning_rate = 0.2")
plt.legend()
plt.title(label = "How much say each tree gets in the final vote")
plt.xlabel(xlabel = "Order the tree was added (1 = first, 100 = last)")
plt.ylabel(ylabel = "Tree's weight in the final vote")
plt.show()

Looking at the plot, notice how the learning rate changes how much each tree contributes:

- With a lower learning rate (orange), the weight is spread out: more of the trees make noticeable contributions to the ensemble
- With a higher learning rate (blue), the earliest trees do most of the work and later trees get almost no weight

There is an inherent tradeoff between the learning rate and the number of boosting iterations (i.e., the number of trees):

- Smaller learning rates should be paired with more boosting iterations so that the ensemble can continue to improve
- Larger learning rates don't need as many boosting iterations

Let's study the implications of using different learning rates for the same number of boosting iterations i.e. the same number of trees. In this code block we'll fix the number of trees at 100, then:

- Consider many different learning rates
- For each learning rate
    - Fit an AdaBoost model with 100 trees
    - Compute the estimated accuracy using CV
- Plot to compare them


In [ ]:
from sklearn.model_selection import cross_val_score

# Different learning rates to consider
rates_to_try = [0.01, 0.1, 0.2, 0.5, 1, 1.5, 2]
# Save the CV estimated accuracies here
cv_result = []

for rate in rates_to_try:
    # Fit ADA
    ada_model_temp = AdaBoostClassifier(n_estimators = 100, learning_rate = rate, random_state = 76)
    
    # Computed estimated accuracy using CV
    cv_scores = cross_val_score(estimator = ada_model_temp, X = train_X, y = train_y, cv = 5, scoring = 'accuracy')
    cv_score_avg = np.average(cv_scores)

    # Save estimated accuracy
    cv_result.append(cv_score_avg)
        
# Plot of CV accuracy
plt.plot(rates_to_try, cv_result, marker = "o")
plt.title(label = "Cross-validated accuracy by learning rate (100 trees)")
plt.xlabel(xlabel = "Learning rate")
plt.ylabel(ylabel = "Cross-validated accuracy")
plt.show()

Learning rates that are too small lead to biased models that never fully learn all of the patterns present in the data, while learning rates that are too large lead to overfit models.

Now let's see what happens when we do the opposite: We fix the learning rate at 0.5 (what seemed to work best in the exploration above) and vary the number of boosting iterations:

In [ ]:
# Different number of boosting iterations i.e. trees to consider
n_iter_to_try = [10,100,200,300,450,600]
# Save the CV estimated accuracies here
cv_result = []

for it in n_iter_to_try:
    ada_model_temp = AdaBoostClassifier(n_estimators = it, learning_rate = 0.5, random_state = 76)
    
    # Computed estimated accuracy using CV
    cv_scores = cross_val_score(estimator = ada_model_temp, X = train_X, y = train_y, cv = 5, scoring = 'accuracy')
    cv_score_avg = np.average(cv_scores)

    # Save estimated accuracy
    cv_result.append(cv_score_avg)

# Plot of CV accuracy
plt.plot(n_iter_to_try, cv_result, marker = 'o')
plt.title(label = "Cross-validated accuracy by number of trees (learning rate 0.5)")
plt.xlabel(xlabel = "Number of Trees")
plt.ylabel(ylabel = "Cross-validated accuracy")
plt.show()

We can see that there's a clear relationship between these two tuning parameters. However, it's difficult to assess what is a large or a small learning rate (or a large or small number of boosting iterations) from one application to another. 

## B) Question 1
    
- **Part a)**: Use `GridSearchCV()` to evaluate the cross-validated accuracy of each combination of the following:
    - Learning rate: 0.1, 0.3, or 0.6
    - Number of boosting iterations: 10, 100, 200, or 400
- **Part b)**: Using the learning rate of the best performing model identified in Part a), create a line chart showing the cross-validated accuracy at various numbers of boosting iterations that surround number iterations identified in your search from Part a). You should choose values so that your graph displays a clear peak/top.
- **Part c)**: By default `AdaBoostClassifier()` will use decision trees with a maximum depth of 1 as base models, but you can change this using the `estimator` argument (ie: `estimator = DecisionTreeClassifier(max_depth = 2)` uses decision trees with a max depth of 2 as base models). Add a second line to your line chart in Part b) that shows the performance of decision trees with a max depth of 2 for the same tuning parameter values you used in Part b).

In [ ]:
# Code for Q1.a)
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':  [10, 100, 200, 400],  # Number of boosting iterations
    'learning_rate': [0.1, 0.3, 0.6]  # Learning rates
}

# Initialize the AdaBoostClassifier
abc = AdaBoostClassifier(random_state = 76)

# GridSearchCV
grid_search = GridSearchCV(estimator = abc, param_grid = param_grid, cv = 5, scoring = 'accuracy').fit(X = train_X, y = train_y)

# Print values
print(grid_search.best_params_)
print(grid_search.best_score_)

In [ ]:
# Code for Q1.b)
# The best rate above was 0.3
# Save values here
cv_result_depth1 = []

n_iter_to_try = [50,100,150,200,250,400,450]
for it in n_iter_to_try:
    ada_model_temp = AdaBoostClassifier(
        n_estimators = it, 
        learning_rate = 0.3,
        random_state = 76
    )
    cv_result_depth1.append(np.average(cross_val_score(estimator = ada_model_temp, X = train_X, y = train_y, cv = 5)))

# Plot of CV accuracy
plt.plot(n_iter_to_try, cv_result_depth1, marker = 'o')
plt.title(label = "Cross-validated accuracy by number of trees (learning rate 0.3)")
plt.xlabel(xlabel = "Number of Trees")
plt.ylabel(ylabel = "Cross-validated accuracy")
plt.show()

In [ ]:
# Code for Q1.c)
from sklearn.tree import DecisionTreeClassifier

cv_result_depth2 = []
n_iter_to_try = [50,100,150,200,250,400,450]
for it in n_iter_to_try:
    ada_model_temp = AdaBoostClassifier(
        estimator = DecisionTreeClassifier(max_depth = 2, random_state = 76), 
        n_estimators = it, 
        learning_rate = 0.3,
        random_state = 76
    )
    cv_result_depth2.append(np.average(cross_val_score(estimator = ada_model_temp, X = train_X, y = train_y, cv = 5)))

# Plot of CV accuracy
plt.plot(n_iter_to_try, cv_result_depth1, marker = 'o', label = "Max depth 1 (default)")
plt.plot(n_iter_to_try, cv_result_depth2, marker = 'o', label = "Max depth 2")
plt.title(label = "Cross-validated accuracy by number of trees (learning rate 0.3)")
plt.xlabel(xlabel = "Number of Trees")
plt.ylabel(ylabel = "Cross-validated accuracy")
plt.legend(title = "Base model")
plt.show()

## C) `xgboost`

The explorations in Section A of the lab illustrate the tradeoff between the two most important hyperparameters in boosting ensembles, the learning rate and the number of boosting iterations.  The motivation behind the `xgboost` library was to provide an efficient implementation of gradient boosting with numerous hyperparameters that could be customized.

Additionally, `xgboost` was designed to, among other things, be able to run on a GPU using parallelization. These features have made `xgboost` widely used, though it is fundamentally just an implementation of gradient boosting.

When training an `xgboost` model, you should carefully consider the following hyperparameters (listed roughly in their order of importance):

- `learning_rate` - The shrinkage applied to estimates at each boosting iteration that controls how much each additional boosting iteration contributes to the ensemble. The learning rate can be set to any value in $(0,1]$.
- `n_estimators` - The number of boosting iterations (ie: trees in the ensemble). More boosting iterations tends to work best when combined with a smaller learning rate (and vice-versa).

There are several parameters that influence the individual base models within the ensemble, here is one of the more intuitive ones

- `max_depth` - The maximum depth of trees. Higher values increase the likelihood of overfitting but provide increased flexibility. Generally maximum depths of 1 or 2 tend to work best in boosting ensembles.

Fortunately for us, `xgboost` is compatible with `sklearn` pipelines and grid searches:

In [ ]:
from sklearn.pipeline import Pipeline 
from xgboost import XGBClassifier

# Example pipeline
pipe = Pipeline(steps = [
    ('model', XGBClassifier(eval_metric = 'error', random_state = 76))
])

# Try two different learning rates
params = {'model__learning_rate': [0.01, 0.1, 0.3]}

# Grid Search
grid_result = GridSearchCV(
    estimator = pipe, 
    param_grid = params, 
    cv = 5
    ).fit(X = train_X, y = train_y)

# Print best hyperparameters and best accuracy
print(grid_result.best_params_)
print(grid_result.best_score_)

Finally

1. While we will not use it in this lab you should recognize that the `XGBRegressor()` function should be used when predicting a numeric outcome
1. You can later use the pipe with the parameters set to the optimal values that yield the best accuracy, here `learning_rate` = 0.01 yielding 63.4% accuracy via `grid_result.best_estimator_`

## D) Question 2

For this question you should use the well-switching data used throughout this lab.

- **Part a)**: Create a data processing pipeline that includes a min-max scaling step followed by an `xgboost` classification model.
- **Part b)**: Using your pipeline from Part a) use `GridSearchCV()` to explore the cross-validated classification accuracy that results from
    - a learning rate of 0.01 
    - Base models having maximum depths of 1 thru 10.
    - Default values for every other tuning parameter
 - **Part c)**: Display the relationship between maximum depth and cross-validated accuracy using a line chart.
- **Part d)**: Using the best maximum depth from Part b), perform 10 iterations of randomized search using a [uniform distribution](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.uniform.html) to draw values of the `learning_rate` parameter between $(0,1]$ and draw values of the `n_estimators` parameter between 100 and 500. Save the results of the `RandomizedSearchCV()` in `random_search`
- **Part e)**: Evaluate the best model from Part d) *on the test set*. Report its classification accuracy and F1 score. Recall you can apply the model with the best parameters that yields the best cross-validated scores to obtain predictions with `random_search.best_estimator_.predict()`

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# Code for Q2.a)
# Part a)
pipe_q2 = Pipeline(steps = [
    ('scaling', MinMaxScaler()),
    ('xgb', XGBClassifier(eval_metric = 'error', random_state = 76))
])

In [ ]:
# Code for Q2.b)
param_grid = {
    'xgb__max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'xgb__learning_rate': [0.01]
}
grid_search = GridSearchCV(
    estimator = pipe_q2, 
    param_grid = param_grid, 
    cv = 5, 
    scoring = 'accuracy')
grid_search.fit(X = train_X, y = train_y)

In [ ]:
# Code for Q2.c)
# Display the relationship between maximum depth and cross-validated accuracy
plt.plot(param_grid['xgb__max_depth'], grid_search.cv_results_['mean_test_score'], marker = 'o')
plt.xlabel(xlabel = 'Maximum Tree Depth')
plt.ylabel(ylabel = 'Cross-Validation Estimated Accuracy')
plt.show()

In [ ]:
# Starter code for Q2.d)
best_max_depth = grid_search.best_params_['xgb__max_depth']

from scipy.stats import uniform, randint
param_dist = {
    'xgb__max_depth': [best_max_depth],
    'xgb__learning_rate': uniform(loc = 0, scale = 1),
    'xgb__n_estimators': randint(low = 100, high = 500)
}

In [ ]:
# Code for Q2.d)
random_search = RandomizedSearchCV(
    estimator = pipe_q2, 
    param_distributions = param_dist, 
    cv = 5, 
    scoring = 'accuracy', 
    n_iter = 10,
    random_state = 76
    ).fit(X = train_X, y = train_y)

# Print best values
print(random_search.best_params_)
print(random_search.best_score_)

In [ ]:
# Code for Q2.e)
# Predict on the test set using the best model from the randomized search
test_y_pred = random_search.best_estimator_.predict(X = test_X)

print("Test accuracy: " + str(accuracy_score(y_true = test_y, y_pred = test_y_pred)))
print("Test F1 score: " + str(f1_score(y_true = test_y, y_pred = test_y_pred)))


Compare the test accuracy to the baseline of always predicting `'yes'` (about 57.5% of households, per the histogram above). A model that barely beats this baseline has learned very little from the features.